# Codex-powered OPAL pilot

The GRU architecture remains unchanged. OPAL changes optimizer programs during training.
The proposed language model is gpt-6.1-sol with high effort.
Use the optional setup on a fresh machine. Preflight never starts an agent.
All training cells are skipped during report export, even when flags are enabled.
Read docs/opal-pilot.md before enabling any training flag.


In [ ]:
import json
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("gru_suite_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/gru-opal-codex.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()
REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "experiment/opal-codex-pilot"  # Select the PR branch before merge
RUN_SETUP = False
RUN_KEEP = False
RUN_CONTROL = False
RUN_LIVE = False
logger.info("Repository: %s", ROOT)
logger.info("Run setup: %s", RUN_SETUP)
logger.info("Keep training enabled: %s", RUN_KEEP)
logger.info("Control training enabled: %s", RUN_CONTROL)
logger.info("Live training enabled: %s", RUN_LIVE)

## Optional setup

In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook --group opal --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")


## Preflight without optimization

Validate data and tensor shapes. This does not check Codex authentication or launch OPAL.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook --group opal python -m sleep_classifier.gru_training \
    --config configs/gru-opal-codex.yaml --check
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Pilot preflight failed")

## Fresh Adam control

Enable RUN_CONTROL only when ready. Use a fresh output directory for another run.

In [ ]:
if RUN_CONTROL:
    !uv run --locked --group gru --group notebook --group opal python -u \
        -m sleep_classifier.gru_training \
        --config configs/gru-opal-control.yaml --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Fresh Adam control failed")

## Offline keep comparison

Enable RUN_KEEP only when ready. Use a fresh output directory for another run.

In [ ]:
if RUN_KEEP:
    !uv run --locked --group gru --group notebook --group opal python -u \
        -m sleep_classifier.gru_training \
        --config configs/gru-opal-keep.yaml --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Offline keep comparison failed")

## Check the Codex runtime before live training

Run this cell manually. It checks executable discovery, version, and the isolated login without inference.
If login is missing, run the printed command in Terminal, then rerun this check.
This cell is skipped during report export. The app-bundled CLI is used when codex is absent from PATH.
Set CODEX_BINARY or optimizer_agent.binary to select another executable.

In [ ]:
!uv run --locked --group gru --group notebook --group opal python \
    -m sleep_classifier.gru_training --config configs/gru-opal-codex.yaml --check-agent
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Complete the printed Codex setup before live training")

## Live Codex pilot

Enable RUN_LIVE only when ready. Use a fresh output directory for another run.

In [ ]:
if RUN_LIVE:
    !uv run --locked --group gru --group notebook --group opal python -u \
        -m sleep_classifier.gru_training \
        --config configs/gru-opal-codex.yaml --train --allow-live-agent
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Live Codex pilot failed")

## Inspect saved metrics

Inspect OPAL programs, applied decisions, wait times, and token counts before interpreting any live score.

In [ ]:
import pandas as pd
import yaml

records = []
for recipe in ["gru-opal-control", "gru-opal-keep", "gru-opal-codex"]:
    config_path = ROOT / f"configs/{recipe}.yaml"
    config = yaml.safe_load(config_path.read_text())
    metrics_path = (config_path.parent / config["output_dir"] / "metrics.json").resolve()
    if metrics_path.exists():
        metrics = json.loads(metrics_path.read_text())
        records.append({"recipe": recipe, "macro_f1": metrics["macro_f1"],
                        "accuracy": metrics["accuracy"], "best_epoch": metrics["best_epoch"]})
        logger.info("Optimizer provenance: %s", metrics.get("optimizer"))
        display(pd.DataFrame(metrics["classes"]).T)
if records:
    display(pd.DataFrame(records))
else:
    logger.info("No pilot results exist. Preflight did not train.")

## Export HTML

In [ ]:
!uv run --locked --group gru --group notebook --group opal python scripts/execute_notebook.py \
    --notebook notebooks/10_opal_pilot.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Report export failed")